In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from labcore.data.datadict_storage import datadict_from_hdf5


def inspect_measurement_keys(path):
    path = Path(path)
    data = datadict_from_hdf5(path)
    print("Measurement-like keys:")
    for k in ["current", "signal", "ssb_frequency"]:
        if k in data:
            try:
                arr = np.asarray(data[k]["values"])
                print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")
            except Exception as e:
                print(f"{k}: failed to read -> {e}")
        else:
            print(f"{k}: not found")
    return data


def load_res_spec_ddh5(path):
    path = Path(path)
    data = datadict_from_hdf5(path)

    current = np.asarray(data["current"]["values"])
    signal = np.asarray(data["signal"]["values"])
    freq = np.asarray(data["ssb_frequency"]["values"])
    print("Raw shapes:")
    print("  current:", current.shape)
    print("  signal:", signal.shape)
    print("  ssb_frequency:", freq.shape) #Following read data file can be changed due to different savinf stiles

    # ----------------------------
    # Case 1: signal is already 2D = (n_flux, n_freq)
    # ----------------------------
    if signal.ndim == 2:
        signal2d = signal

        if current.ndim == 1:
            flux_axis = current.astype(float)
        elif current.ndim == 2:
            flux_axis = current[:, 0].astype(float)
        else:
            raise ValueError(f"Unsupported current shape: {current.shape}")

        if freq.ndim == 1:
            freq_axis = freq.astype(float)
        elif freq.ndim == 2:
            freq_axis = freq[0, :].astype(float)
        else:
            raise ValueError(f"Unsupported ssb_frequency shape: {freq.shape}")

    # ----------------------------
    # Case 2: signal has repetition axis = (n_rep, n_flux, n_freq)
    # ----------------------------
    elif signal.ndim == 3:
        signal2d = np.mean(signal, axis=0)

        if current.ndim == 1:
            flux_axis = current.astype(float)
        elif current.ndim == 2:
            flux_axis = current[0, :, 0].astype(float) if current.shape[-1] > 1 else current[0, :].astype(float)
        elif current.ndim == 3:
            flux_axis = current[0, :, 0].astype(float)
        else:
            raise ValueError(f"Unsupported current shape: {current.shape}")

        if freq.ndim == 1:
            freq_axis = freq.astype(float)
        elif freq.ndim == 2:
            freq_axis = freq[0, :].astype(float)
        elif freq.ndim == 3:
            freq_axis = freq[0, 0, :].astype(float)
        else:
            raise ValueError(f"Unsupported ssb_frequency shape: {freq.shape}")
        print("Detected repetition axis -> averaged over axis 0")
    else:
        raise ValueError(f"Unsupported signal shape: {signal.shape}")

    print("\nProcessed shapes:")
    print("  flux_axis:", flux_axis.shape)
    print("  freq_axis:", freq_axis.shape)
    print("  signal2d:", signal2d.shape)
    return flux_axis, freq_axis, signal2d


def plot_res_spec_magnitude(flux_axis, freq_axis, signal2d):
    mag = np.abs(signal2d)
    plt.figure(figsize=(7, 5))
    plt.pcolormesh(flux_axis, freq_axis, mag.T, shading="auto")
    plt.xlabel("Flux / Current")
    plt.ylabel("SSB Frequency")
    plt.title("Resonator spectroscopy vs flux (magnitude)")
    plt.colorbar(label="|signal|")
    plt.tight_layout()
    plt.show()

In [ ]:
path = r"data/FFM_Set_2_F1.ddh5" #change to data file path in your computer starting from file name parallel to this notebook

data = inspect_measurement_keys(path)
flux_axis, freq_axis, signal2d = load_res_spec_ddh5(path)
plot_res_spec_magnitude(flux_axis, freq_axis, signal2d)

In [ ]:
#This cell fits and select good data points of the resonator spec vs flux and define helper functions for future use
from cqedtoolbox.protocols.operations.fluxonium.res_spec_vs_flux import import add_mag_and_unwind_and_choose_fit

def fit_single_resonator_vs_flux(flux_axis,freq_axis,signal2d,snr_threshold=2.0):
    flux_axis = np.asarray(flux_axis, dtype=float)
    freq_axis = np.asarray(freq_axis, dtype=float)
    signal2d = np.asarray(signal2d)
    fr_fit = []
    snr_list = []
    fit_ok = []

    for i in range(signal2d.shape[0]):
        sig_row = signal2d[i, :]
        try:
            ret = add_mag_and_unwind_and_choose_fit(freq_axis, sig_row, model_choice="single")
            p = ret.fit_result.params
            fr0 = float(p["f_0"])
            snr = float(ret.snr)

            fr_fit.append(fr0)
            snr_list.append(snr)
            fit_ok.append(np.isfinite(fr0) and np.isfinite(snr))
        except Exception as e:
            fr_fit.append(np.nan)
            snr_list.append(np.nan)
            fit_ok.append(False)
            print(f"Row {i} fit failed: {e}")

    fr_fit = np.asarray(fr_fit, dtype=float)
    snr_list = np.asarray(snr_list, dtype=float)
    fit_ok = np.asarray(fit_ok, dtype=bool)
    good_mask = fit_ok & np.isfinite(fr_fit) & np.isfinite(snr_list) & (snr_list >= snr_threshold)
    return fr_fit, snr_list, good_mask


def estimate_period_autocorr(x, y):
    """
    Estimate one dominant period in x from oscillatory y(x).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    y0 = y - np.nanmean(y)
    y0 = np.nan_to_num(y0, nan=0.0)

    n = len(y0)
    if n < 8:
        return x[-1] - x[0]

    dx = float(np.mean(np.diff(x)))
    nfft = 1 << ((2 * n - 1).bit_length())
    fy = np.fft.rfft(y0, nfft)
    ac = np.fft.irfft(fy * np.conj(fy))[:n].real

    if ac[0] != 0:
        ac = ac / ac[0]
    lag_axis = np.arange(n) * dx
    # ignore zero lag
    search = slice(1, n // 2 if n // 2 > 2 else n)
    idx = np.argmax(ac[search]) + 1
    return lag_axis[idx]


def choose_one_period_window(flux_good, fr_good, period_est):
    """
    Choose one period window inside the valid region.
    For now: start at min valid flux and take one estimated period.
    """
    start = float(flux_good.min())
    end = start + float(period_est)

    # if that exceeds range, shift back
    if end > flux_good.max():
        end = float(flux_good.max())
        start = end - float(period_est)

    return start, end


def resample_one_period(flux_seg, fr_seg, n_points=121):
    flux_seg = np.asarray(flux_seg, dtype=float)
    fr_seg = np.asarray(fr_seg, dtype=float)

    order = np.argsort(flux_seg)
    flux_seg = flux_seg[order]
    fr_seg = fr_seg[order]

    x_new = np.linspace(flux_seg.min(), flux_seg.max(), n_points, endpoint=False)
    y_new = np.interp(x_new, flux_seg, fr_seg)

    return x_new, y_new


def running_median(x, window=21):
    x = np.asarray(x, dtype=float)
    n = len(x)
    half = window // 2
    out = np.empty(n, dtype=float)

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        out[i] = np.nanmedian(x[lo:hi])
    return out


def robust_filter_fitted_curve(flux_axis,fr_fit,snr_list,freq_axis,snr_threshold=2.0,median_window=21,abs_dev_threshold=None,mad_multiplier=6.0):
    flux_axis = np.asarray(flux_axis, dtype=float)
    fr_fit = np.asarray(fr_fit, dtype=float)
    snr_list = np.asarray(snr_list, dtype=float)
    freq_axis = np.asarray(freq_axis, dtype=float)

    basic_mask = (
        np.isfinite(flux_axis)
        & np.isfinite(fr_fit)
        & np.isfinite(snr_list)
        & (snr_list >= snr_threshold)
    )
    fmin = float(np.min(freq_axis))
    fmax = float(np.max(freq_axis))
    fspan = fmax - fmin
    margin = 0.05 * fspan
    range_mask = (fr_fit >= fmin - margin) & (fr_fit <= fmax + margin)
    mask1 = basic_mask & range_mask
    fr_tmp = fr_fit.copy()
    fr_tmp[~mask1] = np.nan

    local_med = running_median(fr_tmp, window=median_window)
    dev = np.abs(fr_fit - local_med)

    mad = np.nanmedian(np.abs(fr_tmp - local_med))
    if not np.isfinite(mad) or mad < 1e-12:
        mad = 1e-12
    if abs_dev_threshold is None:
        abs_dev_threshold = mad_multiplier * mad
    
    mask2 = dev <= abs_dev_threshold
    final_mask = mask1 & mask2

    print("Filter summary:")
    print("  total points         =", len(fr_fit))
    print("  pass basic validity  =", int(np.sum(mask1)))
    print("  pass local-median    =", int(np.sum(final_mask)))
    print("  MAD                  =", mad)
    print("  abs_dev_threshold    =", abs_dev_threshold)
    return final_mask, local_med, dev


fr_fit, snr_list, good_mask = fit_single_resonator_vs_flux(flux_axis,freq_axis,signal2d,snr_threshold=2.0)

good_mask2, local_med, dev = robust_filter_fitted_curve(
    flux_axis=flux_axis,
    fr_fit=fr_fit,
    snr_list=snr_list,
    freq_axis=freq_axis,
    snr_threshold=2.0,     # can raise to 3 or 5 later
    median_window=16,      # larger window = smoother local reference
    abs_dev_threshold=None,
    mad_multiplier=20.0,
)

%matplotlib inline
# ---- plot result on raw map ----
plt.figure(figsize=(7, 5))
plt.pcolormesh(flux_axis, freq_axis, np.abs(signal2d).T, shading="auto")
plt.plot(flux_axis[good_mask2], fr_fit[good_mask2], "r.-", ms=3, label="filtered fit")
plt.xlabel("Flux / Current")
plt.ylabel("SSB Frequency")
plt.title("Raw map with robustly filtered fitted readout frequency")
plt.colorbar(label="|signal|")
plt.legend()
plt.tight_layout()
plt.show()

# ---- plot fitted points only ----
plt.figure(figsize=(7, 4))
plt.scatter(flux_axis, fr_fit, s=4, alpha=0.2, label="all fits")
plt.scatter(flux_axis[good_mask2], fr_fit[good_mask2], s=8, color="red", label="kept points")
plt.plot(flux_axis, local_med, "k-", lw=1.5, label="local median")
plt.xlabel("Flux / Current")
plt.ylabel("Fitted Readout Frequency")
plt.title("Robust filtering of fitted curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# This cell is for auto period picking try if you have >1.5 periods (best if >2 periods) and save the data points for ML testing
# Auto period estimation + auto one-period crop
# Do not run the next picking period by hand cell if you run this

N_CNN = 121 # This should be set to the same as your training grids used for ML
save_path = "exp_res_spec_period_for_cnn_test_auto.npz" # Where you want to save the fitted data used for ML offset test

def infer_period_from_curve(flux_good, fr_good, n_uniform=2000):
    """
    Infer the period from filtered fitted fr(flux).
    Calculating auto correlation of original fitting line with a shifted one
    This avoids picking tiny wiggles by only allowing periods that imply
    a reasonable number of periods across the measured data.
    """

    flux_good = np.asarray(flux_good, dtype=float)
    fr_good = np.asarray(fr_good, dtype=float)
    order = np.argsort(flux_good)
    x = flux_good[order]
    y = fr_good[order]
    x_min, x_max = float(x.min()), float(x.max())
    width = x_max - x_min
    xu = np.linspace(x_min, x_max, n_uniform)
    yu = np.interp(xu, x, y)
    p = np.polyfit(xu, yu, 1)
    yu0 = yu - np.polyval(p, xu)
    yu0 = yu0 - np.mean(yu0)

    # Autocorrelation
    n = len(yu0)
    fft = np.fft.rfft(yu0, n=2*n)
    ac = np.fft.irfft(fft * np.conj(fft))[:n]
    ac = ac / ac[0]
    lags = np.arange(n) * (xu[1] - xu[0])

    min_period = width / 4.0
    max_period = width * 0.95
    mask = (lags >= min_period) & (lags <= max_period)
    if not np.any(mask):
        raise ValueError("No valid period range found.")
    search_lags = lags[mask]
    search_ac = ac[mask]
    best_idx = int(np.argmax(search_ac))
    period_est = float(search_lags[best_idx])

    print("Auto period estimate:")
    print("  data width      =", width)
    print("  allowed range   =", (min_period, max_period))
    print("  estimated period=", period_est)
    print("  approx periods  =", width / period_est)
    return period_est, xu, ac, lags


def score_one_period_window(flux_good, fr_good, x0, x1, n_points=121):
    """
    Score a proposed one-period window (where/which point to start with the period we picked).
    Lower score is better.

    We prefer:
    - enough fitted points
    - no large gaps
    - full coverage of window
    - not extremely jagged after interpolation
    """

    mask = (flux_good >= x0) & (flux_good <= x1)
    fx = flux_good[mask]
    fy = fr_good[mask]
    if len(fx) < 30:
        return np.inf
    order = np.argsort(fx)
    fx = fx[order]
    fy = fy[order]
    width = x1 - x0
    if width <= 0:
        return np.inf
    gaps = np.diff(fx)
    max_gap = np.max(gaps) if len(gaps) else np.inf
    gap_score = max_gap / width
    coverage = (fx.max() - fx.min()) / width
    coverage_penalty = abs(1.0 - coverage)
    x_new = np.linspace(fx.min(), fx.max(), n_points, endpoint=False)
    y_new = np.interp(x_new, fx, fy)
    roughness = np.std(np.diff(y_new, n=2))
    score = roughness + 1e7 * gap_score + 1e7 * coverage_penalty
    return score


def auto_pick_one_period_window(flux_good, fr_good, n_points=121, n_start_trials=200):
    """
    Fully automatic:
    infer period, slide a window of that width, choose best window.
    """

    flux_good = np.asarray(flux_good, dtype=float)
    fr_good = np.asarray(fr_good, dtype=float)
    order = np.argsort(flux_good)
    flux_good = flux_good[order]
    fr_good = fr_good[order]

    period_est, xu, ac, lags = infer_period_from_curve(flux_good, fr_good)
    data_min = float(flux_good.min())
    data_max = float(flux_good.max())

    if data_max - data_min < period_est:
        raise ValueError("Data span is smaller than inferred one period.")

    starts = np.linspace(data_min, data_max - period_est, n_start_trials)
    scores = []
    for x0 in starts:
        x1 = x0 + period_est
        s = score_one_period_window(flux_good, fr_good, x0, x1, n_points=n_points)
        scores.append(s)

    scores = np.asarray(scores)
    best_idx = int(np.nanargmin(scores))
    x_min = float(starts[best_idx])
    x_max = float(x_min + period_est)

    print("\nAuto-selected crop:")
    print("  x_min =", x_min)
    print("  x_max =", x_max)
    print("  width =", x_max - x_min)
    print("  score =", scores[best_idx])
    return x_min, x_max, period_est, starts, scores, lags, ac


flux_good = np.asarray(flux_axis[good_mask2], dtype=float)
fr_good = np.asarray(fr_fit[good_mask2], dtype=float)
order = np.argsort(flux_good)
flux_good = flux_good[order]
fr_good = fr_good[order]

x_min, x_max, period_est, starts, scores, lags, ac = auto_pick_one_period_window(flux_good,fr_good,n_points=N_CNN,n_start_trials=200)

crop_mask = (flux_good >= x_min) & (flux_good <= x_max)
flux_crop = flux_good[crop_mask]
fr_crop = fr_good[crop_mask]
print("\nChosen crop:")
print("  number of kept fitted points =", len(flux_crop))
if len(flux_crop) < 10:
    raise ValueError("Too few points in auto-chosen crop window.")


flux_resampled, fr_resampled = resample_one_period(flux_crop,fr_crop,n_points=N_CNN)
plt.figure(figsize=(7, 3))
plt.plot(lags, ac, "-")
plt.axvline(period_est, color="red", linestyle="--", label="estimated period")
plt.xlabel("Lag in flux/current")
plt.ylabel("Autocorrelation")
plt.title("Autocorrelation used to infer period")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.plot(starts, scores, "o-", ms=3)
plt.axvline(x_min, color="red", linestyle="--", label="chosen start")
plt.xlabel("Window start flux/current")
plt.ylabel("Window score")
plt.title("Auto crop score vs window start")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.pcolormesh(flux_axis, freq_axis, np.abs(signal2d).T, shading="auto")
plt.plot(flux_good, fr_good, "r.", ms=2, alpha=0.35, label="filtered fitted points")
plt.plot(flux_resampled, fr_resampled, "wo", ms=3, mec="k", mew=0.5, label="resampled 121 pts")
plt.axvline(x_min, color="cyan", linestyle="--", label="auto crop window")
plt.axvline(x_max, color="cyan", linestyle="--")
plt.xlabel("Flux / Current")
plt.ylabel("SSB Frequency")
plt.title("Auto-chosen one period and resampled CNN input")
plt.colorbar(label="|signal|")
plt.legend()
plt.tight_layout()
plt.show()

np.savez_compressed(save_path,flux_axis,freq_axis,signal2d,flux_good,fr_good,x_min,x_max,period_est,flux_crop,fr_crop,flux_resampled,fr_resampled)
print(f"Saved to: {save_path}")

In [ ]:
# Picking one period by hand if data's flux range only contains ~1 period

x_min = -0.000082 # Minimum flux to start with
x_max = 0.00016 # Maximum flux that ends
N_CNN = 121 # This should be set to the same as your training grids used for ML
save_path = "exp_res_spec_period_for_cnn_test_auto.npz" # Where you want to save the fitted data used for ML offset test


flux_good = np.asarray(flux_axis[good_mask2], dtype=float)
fr_good = np.asarray(fr_fit[good_mask2], dtype=float)
order = np.argsort(flux_good)
flux_good = flux_good[order]
fr_good = fr_good[order]
crop_mask = (flux_good >= x_min) & (flux_good <= x_max)
flux_crop = flux_good[crop_mask]
fr_crop = fr_good[crop_mask]

print("Chosen crop:")
print("  x_min =", x_min)
print("  x_max =", x_max)
print("  chosen period =", x_max-x_min)
print("  number of kept fitted points =", len(flux_crop))

if len(flux_crop) < 10:
    raise ValueError("Too few points in chosen crop window. Widen the window a bit.")

flux_resampled, fr_resampled = resample_one_period(flux_crop, fr_crop, n_points=N_CNN)
print("Resampled length =", len(fr_resampled))

plt.figure(figsize=(8, 5))
plt.pcolormesh(flux_axis, freq_axis, np.abs(signal2d).T, shading="auto")
plt.plot(flux_good, fr_good, "r.", ms=2, alpha=0.4, label="filtered fitted points")
plt.plot(flux_resampled, fr_resampled, "wo", ms=3, mec="k", mew=0.5, label="resampled 121 pts")
plt.axvline(x_min, color="cyan", linestyle="--", label="crop window")
plt.axvline(x_max, color="cyan", linestyle="--")
plt.xlabel("Flux / Current")
plt.ylabel("SSB Frequency")
plt.title("Chosen period and resampled CNN input")
plt.colorbar(label="|signal|")
plt.legend()
plt.tight_layout()
plt.show()

np.savez_compressed(save_path,flux_axis,freq_axis,signal2d,flux_good,fr_good,x_min,x_max,flux_crop,fr_crop,flux_resampled,fr_resampled)
print(f"Saved to: {save_path}")

In [ ]:
# Run it to get a visualization of where zero and half flux points are after getting the flux offset from ML
offset_fraction = 0.7315187089842214   # what you get from ML output

def mark_zero_and_half_flux(x_min,x_max,offset_fraction,flux_axis=None,freq_axis=None,signal2d=None,wrap=True):
    """
    Convert predicted offset fraction back to current/flux positions.

    We define the chosen period window as:
        x_min -> normalized flux 0
        x_max -> normalized flux 1

    If offset_fraction is the predicted zero-flux position in normalized units,
    then:
        zero_current = x_min + offset_fraction * (x_max - x_min)
        half_current = x_min + (offset_fraction + 0.5) * (x_max - x_min)

    If wrap=True, half flux is wrapped back into the chosen period.
    """

    period_current = x_max - x_min
    zero_frac = offset_fraction
    half_frac = offset_fraction + 0.5

    if wrap:
        zero_frac = zero_frac % 1.0
        half_frac = half_frac % 1.0

    zero_current = x_min + zero_frac * period_current
    half_current = x_min + half_frac * period_current

    print("Chosen period:")
    print(f"  x_min = {x_min}")
    print(f"  x_max = {x_max}")
    print(f"  period_current = {period_current}")
    print("\nFlux markers:")
    print(f"  offset_fraction = {offset_fraction}")
    print(f"  zero_flux current = {zero_current}")
    print(f"  half_flux current = {half_current}")

    if flux_axis is not None and freq_axis is not None and signal2d is not None:
        plt.figure(figsize=(8, 5))
        plt.pcolormesh(flux_axis, freq_axis, np.abs(signal2d).T, shading="auto")
        plt.axvline(x_min, color="cyan", linestyle="--", label="chosen period start")
        plt.axvline(x_max, color="cyan", linestyle="--", label="chosen period end")
        plt.axvline(zero_current, color="red", linestyle="-", linewidth=2, label="zero flux")
        plt.axvline(half_current, color="orange", linestyle="-", linewidth=2, label="half flux")
        plt.xlabel("Current / Flux")
        plt.ylabel("SSB Frequency")
        plt.title("Zero and half-flux markers on raw map")
        plt.colorbar(label="|signal|")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return zero_current, half_current

zero_current, half_current = mark_zero_and_half_flux(x_min,x_max,offset_fraction,flux_axis,freq_axis,signal2d,wrap=True)